In [1]:
from typing import Literal, Optional, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from rich import print

class ApprovalState(TypedDict):
    action_details: str
    status: Optional[Literal["pending", "approved", "rejected"]]


def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    # Expose details so the caller can render them in a UI
    decision = interrupt(
        {
            "question": "Approve this action?",
            "details": state["action_details"],
        }
    )

    # Route to the appropriate node after resume
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    return {"status": "rejected"}


builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)

# Use a more durable checkpointer in production
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "approval-123"}}
initial = graph.stream_events(
    {"action_details": "Transfer $500", "status": "pending"},
    config=config,
    version="v3",
)
_ = initial.output  # drive the stream to completion
print(initial.interrupts)  # -> (Interrupt(value={'question': ..., 'details': ...}),)

# Resume with the decision; True routes to proceed, False to cancel
resumed = graph.stream_events(Command(resume=True), config=config, version="v3")
print(resumed.output["status"])

e:\LangGraph\.venv\Lib\site-packages\langgraph\pregel\main.py:3679: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
e:\LangGraph\.venv\Lib\site-packages\langgraph\pregel\main.py:3529: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


[
    Interrupt(
        value={'question': 'Approve this action?', 'details': 'Transfer $500'},
        id='f3b8845a7afbe1d315ac7205e2739331'
    )
]

approved

In [2]:
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class ReviewState(TypedDict):
    generated_text: str


def review_node(state: ReviewState):
    # Ask a reviewer to edit the generated content
    updated = interrupt(
        {
            "instruction": "Review and edit this content",
            "content": state["generated_text"],
        }
    )
    return {"generated_text": updated}


builder = StateGraph(ReviewState)
builder.add_node("review", review_node)
builder.add_edge(START, "review")
builder.add_edge("review", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "review-42"}}
initial = graph.stream_events(
    {"generated_text": "Initial draft"}, config=config, version="v3"
)
_ = initial.output  # drive the stream to completion
print(initial.interrupts)  # -> (Interrupt(value={'instruction': ..., 'content': ...}),)

# Resume with the edited text from the reviewer
final_state = graph.stream_events(
    Command(resume="Improved draft after review"),
    config=config,
    version="v3",
)
print(final_state.output["generated_text"])  # -> "Improved draft after review"


[
    Interrupt(
        value={'instruction': 'Review and edit this content', 'content': 'Initial draft'},
        id='bdf53fb504d96f992349d57110d33b89'
    )
]

Improved draft after review

In [3]:
import sqlite3
import operator
from typing import TypedDict, Annotated, Literal
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langchain.messages import AnyMessage, SystemMessage, ToolMessage


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


@tool
def send_email(to: str, subject: str, body: str):
    """Send an email to a recipient."""

    # Pause before sending; payload surfaces on stream.interrupts when using event streaming
    response = interrupt({
        "action": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "message": "Approve sending this email?",
    })

    if response.get("action") == "approve":
        final_to = response.get("to", to)
        final_subject = response.get("subject", subject)
        final_body = response.get("body", body)

        # Actually send the email (your implementation here)
        print(f"[send_email] to={final_to} subject={final_subject} body={final_body}")
        return f"Email sent to {final_to}"

    return "Email cancelled by user"


model = ChatOllama(model="minimax-m3:cloud").bind_tools([send_email])
tools_by_name = {"send_email": send_email}


def agent_node(state: AgentState):
    # LLM may decide to call the tool; interrupt pauses before sending
    result = model.invoke(state["messages"])
    return {"messages": [result]}

def tool_node(state: AgentState):
    """Performs the tool call"""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

def should_continue(state: AgentState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""
    messages = state["messages"]
    last_message = messages[-1]

    if last_message.tool_calls:
        return "tool_node"
    return END

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tool_node", END])  # Routes to "tools" or END
builder.add_edge("tool_node", "agent")  # Loop back after tools

checkpointer = SqliteSaver(
    sqlite3.connect("tool-approval.db", check_same_thread=False)
)
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "email-workflow"}}
initial = graph.stream_events(
    {
        "messages": [
            {"role": "user", "content": "Send an email to alice@example.com about the meeting"}
        ]
    },
    config=config,
    version="v3",
)
initial.output  # drive the stream to completion
print(initial.interrupts)  # -> (Interrupt(value={'action': 'send_email', ...}),)

# Resume with approval and optionally edited arguments
resumed = graph.stream_events(
    Command(resume={"action": "approve", "subject": "Updated subject"}),
    config=config,
    version="v3",
)
print(resumed.output["messages"][-1])  # -> Tool result returned by send_email

[
    Interrupt(
        value={
            'action': 'send_email',
            'to': 'alice@example.com',
            'subject': 'Re: Meeting',
            'body': 'Hi Alice,\n\nJust a quick note regarding our meeting. Please let me know if you have any 
questions, need to reschedule, or would like to add anything to the agenda.\n\nLooking forward to speaking with 
you.\n\nBest regards',
            'message': 'Approve sending this email?'
        },
        id='b2b1496050965da4252fd85a2d09431e'
    )
]

to=alice@example.com subject=Updated subject body=Hi Alice,

Just a quick note regarding our meeting. Please let me know if you have any questions, need to reschedule, or would
like to add anything to the agenda.

Looking forward to speaking with you.

Best regards

AIMessage(
    content=[
        {
            'type': 'text',
            'text': "✅ **Email sent successfully!**\n\nHere's a summary of what was sent:\n\n- **To:** 
alice@example.com\n- **Subject:** Re: Meeting\n- **Body:** A brief follow-up message inviting questions, 
rescheduling requests, or agenda additions.\n\n---\n\n💡 **Tip:** To avoid sending similar follow-up emails 
repeatedly, try sharing some specific details next time, such as:\n\n- 📅 The meeting's **date and time**\n- 📍 The
**location or video call link**\n- 📝 The **agenda or discussion topics**\n- ✅ Any **action items** or preparation
needed\n\nWould you like me to send a more detailed, personalized email about the meeting? Just provide the 
details!",
            'index': 0
        }
    ],
    additional_kwargs={},
    response_metadata={
        'model': 'minimax-m3',
        'created_at': '2026-06-03T17:18:27.333257063Z',
        'done': True,
        'done_reason': 'stop',
        'total_duration': 4489448826,
        'load_duration': None,
        'prompt_eval_count': None,
        'prompt_eval_duration': None,
        'eval_count': 144,
        'eval_duration': None,
        'logprobs': None,
        'model_name': 'minimax-m3',
        'model_provider': 'ollama',
        'output_version': 'v1'
    },
    id='lc_run--019e8e7e-40f1-7b81-bd08-621c650f906c',
    tool_calls=[],
    invalid_tool_calls=[]
)

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

# 1. Define the State schema
class State(TypedDict):
    name: str
    age: str
    city: str

# 2. Define your exact node using sequential interrupts
def node_a(state: State):
    # ✅ Good: interrupt calls happen in the same order every time
    name = interrupt("What's your name?")
    age = interrupt("What's your age?")
    city = interrupt("What's your city?")

    return {
        "name": name,
        "age": age,
        "city": city
    }

# 3. Build and compile the graph with a memory checkpointer
builder = StateGraph(State)
builder.add_node("node_a", node_a)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

# Checkpointer is mandatory for using `interrupt`
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# 4. Configuration configuration (requires a thread_id for state retention)
config = {"configurable": {"thread_id": "user_session_123"}}

# =====================================================================
# SIMULATING EXECUTION TRACK
# =====================================================================

print("--- STARTING THE GRAPH ---")
# First run: Hits the first interrupt ("What's your name?") and pauses
events = graph.stream(None, config)

# Check the state to see what it's waiting for
state_info = graph.get_state(config)
print(f"state_info :: {state_info}")
if state_info.tasks:
    print(f"\nPaused at Interrupt Payload: {state_info.tasks[0].interrupts[0].value}")


print("\n--- RESUMING WITH NAME ---")
# Second run: Sends the name. Re-runs node_a, fills 'name', pauses at 'age'
events = graph.stream(Command(resume="Alice"), config)
for event in events:
    print(event)


print("\n--- RESUMING WITH AGE ---")
# Third run: Sends the age. Re-runs node_a, fills 'name' and 'age', pauses at 'city'
events = graph.stream(Command(resume="30"), config)
for event in events:
    print(event)


print("\n--- RESUMING WITH CITY ---")
# Final run: Sends the city. Node finishes and returns the dictionary to the state
events = graph.stream(Command(resume="New York"), config)
for event in events:
    print(event)

print("\nFinal State Values:")
print(graph.get_state(config).values)


--- STARTING THE GRAPH ---

state_info :: StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'user_session_123'}}, 
metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

--- RESUMING WITH NAME ---

{'__interrupt__': (Interrupt(value="What's your name?", id='1dd6003f8e398c17d26d1b8b36e1e42a'),)}

--- RESUMING WITH AGE ---

{'__interrupt__': (Interrupt(value="What's your age?", id='1dd6003f8e398c17d26d1b8b36e1e42a'),)}

--- RESUMING WITH CITY ---

{'__interrupt__': (Interrupt(value="What's your city?", id='1dd6003f8e398c17d26d1b8b36e1e42a'),)}

Final State Values:

{}

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

# ==========================================================
# 1️⃣ Define the State schema
# ==========================================================
class State(TypedDict):
    user: dict

# ==========================================================
# 2️⃣ Define the node using a single interrupt for multiple fields
# ==========================================================
def node_a(state: State):
    response = interrupt({
        "question": "Enter user details",
        "fields": ["name", "email", "age"],
        "current_values": state.get("user", {})
    })
    return {"user": response}

# ==========================================================
# 3️⃣ Build the graph and add nodes/edges
# ==========================================================
builder = StateGraph(State)
builder.add_node("node_a", node_a)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

# ==========================================================
# 4️⃣ Setup in-memory checkpointer
# ==========================================================
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# ==========================================================
# 5️⃣ Configuration for state retention
# ==========================================================
config = {"configurable": {"thread_id": "user_session_123"}}

# ==========================================================
# 6️⃣ Start the graph (pauses at interrupt)
# ==========================================================
print("--- STARTING THE GRAPH ---")
events = graph.stream(None, config)

# ==========================================================
# 7️⃣ Safely inspect the current interrupt payload
# ==========================================================
state_info = graph.get_state(config)
if state_info.tasks and len(state_info.tasks) > 0:
    interrupts = state_info.tasks[0].interrupts
    if interrupts and len(interrupts) > 0:
        print(f"\nPaused at interrupt payload:\n{interrupts[0].value}")
    else:
        print("\nNo interrupts found inside the task.")
else:
    print("\nNo tasks found in the current state.")

# ==========================================================
# 8️⃣ Resume with user details (all at once)
# ==========================================================
resume_payload = {
    "name": "Alice",
    "email": "alice@example.com",
    "age": "30"
}

print("\n--- RESUMING WITH USER DETAILS ---")
events = graph.stream(Command(resume=resume_payload), config)

for event in events:
    print(event)

# ==========================================================
# 9️⃣ Print final state
# ==========================================================
print("\nFinal State Values:")
print(graph.get_state(config).values)

--- STARTING THE GRAPH ---

No tasks found in the current state.

--- RESUMING WITH USER DETAILS ---

{
    '__interrupt__': (
        Interrupt(
            value={'question': 'Enter user details', 'fields': ['name', 'email', 'age'], 'current_values': {}},
            id='169002fb53d8f5d954016f6212fb1a31'
        ),
    )
}

Final State Values:

{}